# Function 4 Analysis - Week 8

**Function description:** Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

**New datapoint (Week 8):** `0.440000-0.390769-0.330000-0.410000` returned **≈−0.05018**, far below the current max `0.430300-0.359300-0.351800-0.383700` at **≈0.55184**. Total observations: **38**.

**Why we chose this point:** It followed last week’s EI pick just outside the incumbent. The sharp drop confirms the peak is extremely narrow; pushing x3 lower and x4 higher hurts.

**Recommendation for next week:** Stay tightly around the incumbent `(0.43, 0.36, 0.35, 0.38)` with local EI plus drift/diversity; keep x3 ≈0.34–0.37 and x4 ≈0.37–0.40, add light jitter so we don’t re-sample the same point, and favour candidates that remain close while probing the immediate ridge. Replicate near the max if you need to confirm stability versus noise.


## Loading and Displaying the Data

We load the inputs and outputs for function 4. The Week 8 run `(0.440000, 0.390769, 0.330000, 0.410000)` returned **≈−0.05018**, so the max remains the Week 5 point `(0.430300, 0.359300, 0.351800, 0.383700)` at **≈0.5518**, ahead of the Week 3 point `(0.4262, 0.4527, 0.3919, 0.4293)` at ≈−0.0140. Earlier points: Week 1 manual override missed (≈−11.6), Week 2 UCB found ≈−0.058, Week 4 follow-up was ≈−0.100.


In [4]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from IPython.display import display
sns.set_theme(style="ticks", context="notebook")
path = Path("../../initial_data/function_4")
X = np.load(path / "initial_inputs.npy")
y = np.load(path / "initial_outputs.npy")

# Week 1–7 new points
X_new_point_week_1 = np.array([[0.100000, 0.400000, 0.400000, 0.050000]])
y_new_point_week_1 = np.array([-11.551402216263181])
X_new_point_week_2 = np.array([[0.412000, 0.448200, 0.386300, 0.439500]])
y_new_point_week_2 = np.array([-0.05797573871593498])
X_new_point_week_3 = np.array([[0.426200, 0.452700, 0.391900, 0.429300]])
y_new_point_week_3 = np.array([-0.013999616551390925])
X_new_point_week_4 = np.array([[0.430000, 0.455200, 0.393800, 0.427600]])
y_new_point_week_4 = np.array([-0.09998342305973962])
X_new_point_week_5 = np.array([[0.430300, 0.359300, 0.351800, 0.383700]])
y_new_point_week_5 = np.array([0.5518426262369016])
X_new_point_week_6 = np.array([[0.421100, 0.389600, 0.370500, 0.393100]])
y_new_point_week_6 = np.array([0.37109387744135747])
X_new_point_week_7 = np.array([[0.440000, 0.390769, 0.330000, 0.410000]])
y_new_point_week_7 = np.array([-0.05018491923068735])

X = np.vstack([
    X,
    X_new_point_week_1,
    X_new_point_week_2,
    X_new_point_week_3,
    X_new_point_week_4,
    X_new_point_week_5,
    X_new_point_week_6,
    X_new_point_week_7,
])
y = np.concatenate([
    y,
    y_new_point_week_1,
    y_new_point_week_2,
    y_new_point_week_3,
    y_new_point_week_4,
    y_new_point_week_5,
    y_new_point_week_6,
    y_new_point_week_7,
])

df = pd.DataFrame(X, columns=["x1", "x2", "x3", "x4"])
df["y"] = y

# Display original and y-sorted DataFrames side by side
display(df)
print("df sorted by y")
df_sorted = df.sort_values("y", ascending=False).reset_index(drop=True)
df_sorted["x_avg"] = df_sorted[["x1", "x2", "x3", "x4"]].mean(axis=1)
display(df_sorted)


,x1,x2,x3,x4,y
0,0.896981,0.725628,0.175404,0.701694,-22.108288
1,0.889356,0.499588,0.539269,0.508783,-14.601397
2,0.250946,0.033693,0.145380,0.494932,-11.699932
3,0.346962,0.006250,0.760564,0.613024,-16.053765
4,0.124871,0.129770,0.384400,0.287076,-10.069633
5,0.801303,0.500231,0.706645,0.195103,-15.487083
6,0.247708,0.060445,0.042186,0.441324,-12.681685
7,0.746702,0.757092,0.369353,0.206566,-16.026400
8,0.400665,0.072574,0.886768,0.243842,-17.049235
9,0.626071,0.586751,0.438806,0.778858,-12.741766


df sorted by y


,x1,x2,x3,x4,y,x_avg
0,0.430300,0.359300,0.351800,0.383700,0.551843,0.381275
1,0.421100,0.389600,0.370500,0.393100,0.371094,0.393575
2,0.426200,0.452700,0.391900,0.429300,-0.014000,0.425025
3,0.440000,0.390769,0.330000,0.410000,-0.050185,0.392692
4,0.412000,0.448200,0.386300,0.439500,-0.057976,0.421500
5,0.430000,0.455200,0.393800,0.427600,-0.099983,0.426650
6,0.577766,0.428772,0.425826,0.249007,-4.025542,0.420343
7,0.326076,0.472367,0.453192,0.105887,-6.702089,0.339381
8,0.282138,0.505987,0.530531,0.096302,-7.966775,0.353739
9,0.124871,0.129770,0.384400,0.287076,-10.069633,0.231529


- **New point (Week 8):** `(0.440000, 0.390769, 0.330000, 0.410000)` returned **≈−0.05018**, well below the 0.5518 peak at `(0.430300, 0.359300, 0.351800, 0.383700)`.
- Recommendation for next BO step: keep the local EI focus around `(0.43, 0.36–0.39, 0.35–0.40)` with a **narrow box constraint** and a noise term; add a diversity/jitter penalty so proposals don’t snap back exactly to the 0.5518 point but explore its immediate neighborhood (±0.02 per coord).



In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from scipy.optimize import minimize
np.random.seed(42)
kernel = (
    ConstantKernel(1.0, (1e-3, 10.0))
    * Matern(length_scale=[0.25, 0.25, 0.25, 0.25], length_scale_bounds=(0.05, 1.0), nu=1.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-5, 1e-1))
)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=15, random_state=42)
gp.fit(X, y)
print("GP fitted successfully")


GP fitted successfully


d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\OneDrive\Documents\cursor\imperial_college_capstone\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1.0. Increasing the bound and calling fit again may find a bette

## Finding the Next Point to Evaluate (Local EI + diversity)

With the narrow peak confirmed (Week 8 drop to ≈−0.05 just outside the max), we keep a **local EI search** around the incumbent but add:
- A rougher GP with white noise to stay noise-aware.
- Drift penalty on x2/x3 to stay near the ridge.
- Band penalties to keep x3 ≈0.34–0.37 and x4 ≈0.37–0.40.
- A small observation buffer (≥0.01) so we do not re-sample any past point.

This preserves mild exploration (ξ≈0.003) while favouring diverse but tight moves around the max.

In [6]:
# Local EI with drift + diversity penalties (stay near incumbent, avoid repeats)
from scipy.special import erf

best_idx = df["y"].idxmax()
best_point = df.loc[best_idx, ["x1", "x2", "x3", "x4", "y"]]

# Tight local box around the incumbent (matches the recommendation ranges)
x1_grid = np.linspace(0.41, 0.44, 40)
x2_grid = np.linspace(0.34, 0.40, 40)
x3_grid = np.linspace(0.33, 0.38, 40)
x4_grid = np.linspace(0.37, 0.41, 30)
mesh = np.array(np.meshgrid(x1_grid, x2_grid, x3_grid, x4_grid)).reshape(4, -1).T

mu, sigma = gp.predict(mesh, return_std=True)
y_best = y.max()
xi = 0.003  # mild exploration

sigma_safe = np.maximum(sigma, 1e-9)
sqrt2 = np.sqrt(2.0)
sqrt2pi = np.sqrt(2 * np.pi)
z = (mu - y_best - xi) / sigma_safe
ei = (mu - y_best - xi) * 0.5 * (1.0 + erf(z / sqrt2)) + sigma_safe * (np.exp(-0.5 * z**2) / sqrt2pi)
ei[sigma <= 1e-9] = 0.0

# Drift penalty on x2/x3 away from incumbent + penalties on x3/x4 outside the target band
inc_x2x3 = best_point[["x2", "x3"]].to_numpy(dtype=float)
drift = np.linalg.norm(mesh[:, 1:3] - inc_x2x3, axis=1)
band_x3 = (0.34, 0.37)
band_x4 = (0.37, 0.40)
pen_x3 = np.clip(band_x3[0] - mesh[:, 2], 0, None) + np.clip(mesh[:, 2] - band_x3[1], 0, None)
pen_x4 = np.clip(band_x4[0] - mesh[:, 3], 0, None) + np.clip(mesh[:, 3] - band_x4[1], 0, None)
lambda_pen = 0.1
lambda_x3 = 0.2
lambda_x4 = 0.2
penalised_ei = ei - lambda_pen * drift - lambda_x3 * pen_x3 - lambda_x4 * pen_x4

cand = pd.DataFrame(mesh, columns=["x1", "x2", "x3", "x4"])
cand["mu"], cand["sigma"], cand["ei"], cand["drift"], cand["pen_x3"], cand["pen_x4"], cand["ei_pen"] = mu, sigma, ei, drift, pen_x3, pen_x4, penalised_ei

# Diversity: avoid re-sampling any observed point (small buffer); fallback if empty
obs_points = df[["x1", "x2", "x3", "x4"]].to_numpy()
cand["dist_to_obs"] = np.linalg.norm(cand[["x1", "x2", "x3", "x4"]].to_numpy()[:, None, :] - obs_points[None, :, :], axis=2).min(axis=1)
min_obs_buffer = 0.01
cand_feasible = cand[cand["dist_to_obs"] >= min_obs_buffer]
if cand_feasible.empty:
    cand_feasible = cand.copy()

next_point = cand_feasible.loc[cand_feasible["ei_pen"].idxmax()]
print("Current best (observed):", best_point.to_dict())
print("Suggested next query (EI with drift/diversity penalties): {}-{}-{}-{}".format(
    f"{next_point.x1:.6f}", f"{next_point.x2:.6f}", f"{next_point.x3:.6f}", f"{next_point.x4:.6f}"
))
print(
    f"Posterior mean: {next_point.mu:.4f}, std: {next_point.sigma:.4f}, EI: {next_point.ei:.6f}, EI_pen: {next_point.ei_pen:.6f}, drift: {next_point.drift:.4f}, dist_to_obs: {next_point.dist_to_obs:.4f}"
)

# Show top 5 penalised EI candidates
print("\nTop 5 (penalised EI, obs buffer >= 0.01):")
display(
    cand_feasible.nlargest(5, "ei_pen")[
        ["x1", "x2", "x3", "x4", "mu", "sigma", "ei", "ei_pen", "drift", "pen_x3", "pen_x4", "dist_to_obs"]
    ]
)


Current best (observed): {'x1': 0.4303, 'x2': 0.3593, 'x3': 0.3518, 'x4': 0.3837, 'y': 0.5518426262369016}
Suggested next query (EI with drift/diversity penalties): 0.436923-0.340000-0.380000-0.407241
Posterior mean: 0.5808, std: 0.4761, EI: 0.203195, EI_pen: 0.196329, drift: 0.0342, dist_to_obs: 0.0420

Top 5 (penalised EI, obs buffer >= 0.01):


,x1,x2,x3,x4,mu,sigma,ei,ei_pen,drift,pen_x3,pen_x4,dist_to_obs
43197,0.436923,0.34,0.38,0.407241,0.580820,0.476068,0.203195,0.196329,0.034172,0.01,0.007241,0.042021
41997,0.436154,0.34,0.38,0.407241,0.582821,0.473403,0.203179,0.196314,0.034172,0.01,0.007241,0.041907
44397,0.437692,0.34,0.38,0.407241,0.578669,0.478802,0.203164,0.196299,0.034172,0.01,0.007241,0.042149
43198,0.436923,0.34,0.38,0.408621,0.573969,0.485548,0.203419,0.196278,0.034172,0.01,0.008621,0.042809
43196,0.436923,0.34,0.38,0.405862,0.587285,0.466673,0.202846,0.196257,0.034172,0.01,0.005862,0.041264


## Summary and recommended point
- **Current best:** `0.430300-0.359300-0.351800-0.383700` (≈0.5518).
- **Recommended next point (format: 6 decimals, dash-separated):** `0.436923-0.340000-0.380000-0.407241`
- Next 5 alternatives (context, best-first after incumbent):
  - `0.421100-0.389600-0.370500-0.393100` → ≈0.37109
  - `0.426200-0.452700-0.391900-0.429300` → ≈−0.01400
  - `0.440000-0.390769-0.330000-0.410000` → ≈−0.05018
  - `0.412000-0.448200-0.386300-0.439500` → ≈−0.05798
  - `0.430000-0.455200-0.393800-0.427600` → ≈−0.09998

### What changed and why
Rougher GP + white noise; local EI (ξ≈0.003) with drift penalty, x3/x4 band penalties, and a 0.01 observation buffer to avoid repeats. This keeps moves tight around the incumbent but nudges away from already-sampled points.
